In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Magnetic structure fit: Co₂SiO₄ at 2.3 K, D20

Fit the magnetic structure of the olivine Co₂SiO₄ to constant-wavelength neutron powder data
from D20 at ILL, measured at 2.3 K, below its ordering temperature. The cobalt moments order with
k = 0 in the magnetic space group Pnma.1: Co1 may point in any direction, Co2 only along b.

The project is the catalogue case `pd-neut-cwl_cosio-d20_2k`, the one the command line and the
app's Examples open. Its two Co sites share one displacement, and the two FCJ asymmetry terms are
tied, from the start.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Single</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">pd</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">cwl</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal structure</span> <span style="color:#c2185b;background:#c2185b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Magnetic structure</span>

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-cwl_cosio-d20_2k', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['cosio']
experiment = project.experiments['d20']
project.display.constraints()

Data 'pd-neut-cwl_cosio-d20_2k' is at data/pd-neut-cwl_cosio-d20_2k-773039d8ca78/pd-neut-cwl_cosio-d20_2k


View(html=<76,429 characters>, height=264)

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('d20')

View(html=<305,928 characters>, height=480)

## Perform the analysis

The nuclear structure comes first, in the usual order; the magnetic moments come last, once the
nuclear peaks are fitted.

### Scale, 2θ zero and cell

In [6]:
experiment.linked_structures['cosio'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_b.free = True
structure.cell.length_c.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,cosio,cell,,length_a,10.3081,,0,30,Å
2,cosio,cell,,length_b,6.00219,,0,30,Å
3,cosio,cell,,length_c,4.78739,,0,30,Å
4,d20,instrument,,calib_twotheta_offset,0.29433,,-inf,inf,deg
5,d20,linked_structure,cosio,scale,1.81723,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 5 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       14.28        56.27
      1      0.003     14.03        54.28   3.5% ↓
      2      0.007     14.03        54.28   
  done · converged · 2 iters · reduced χ² 54.28 · Rwp 14.03 % · 0.01 s


View(html=<82,724 characters>, height=376)

In [9]:
project.display.pattern('d20')

View(html=<307,013 characters>, height=480)

### Background

With the peaks in place, the sixteen background points follow the level between them.

In [10]:
for point in experiment.background:
    point.intensity.free = True

In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 21 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       14.03        54.89
      1      0.004     13.20        48.60   11.5% ↓
      2      0.009     13.20        48.60   
  done · converged · 2 iters · reduced χ² 48.60 · Rwp 13.20 % · 0.01 s


View(html=<98,476 characters>, height=376)

In [12]:
project.display.pattern('d20')

View(html=<307,198 characters>, height=480)

### Peak widths

In [13]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 25 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.20        48.74
      1      0.005     13.13        48.24   1.0% ↓
      2      0.008     13.13        48.23   
      3      0.013     13.13        48.23   
  done · converged · 3 iters · reduced χ² 48.23 · Rwp 13.13 % · 0.02 s


View(html=<102,772 characters>, height=376)

In [15]:
project.display.pattern('d20')

View(html=<307,315 characters>, height=480)

### Peak asymmetry

The low-angle peaks lean towards low angle. One FCJ term is freed; the other follows it by the
project's constraint.

In [16]:
experiment.peak.asym_fcj_1.free = True

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 26 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.13        48.26
      1      0.041     13.13        48.26   
      2      0.045     13.13        48.26   
      3      0.050     13.13        48.26   
      4      0.054     13.13        48.26   
      5      0.059     13.13        48.26   
      6      0.063     13.13        48.26   
      7      0.067     13.13        48.26   
      8      0.071     13.13        48.26   
      9      0.075     13.13        48.25   
     10      0.079     13.13        48.25   
     11      0.084     13.13        48.25   
     12      0.088     13.13        48.25   
     13      0.094     13.13        48.25   
  done · converged · 13 iters · reduced χ² 48.25 · Rwp 13.13 % · 0.10 s


View(html=<103,899 characters>, height=376)

In [18]:
project.display.pattern('d20')

View(html=<307,333 characters>, height=480)

### Atom positions

Co1 sits on a centre of symmetry and the other sites on the mirror plane, so only their free
coordinates are fitted. The displacements stay at their room-temperature values.

In [19]:
for site in structure.atom_sites:
    for coordinate in (site.fract_x, site.fract_y, site.fract_z):
        coordinate.free = not coordinate.symmetry_constrained

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 37 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.13        48.63
      1      0.007     13.06        48.16   1.0% ↓
      2      0.013     13.06        48.14   
      3      0.020     13.06        48.14   
  done · converged · 3 iters · reduced χ² 48.14 · Rwp 13.06 % · 0.03 s


View(html=<115,301 characters>, height=376)

In [21]:
project.display.pattern('d20')

View(html=<307,322 characters>, height=480)

### Magnetic moments

With the nuclear peaks fitted, the moments fit the magnetic peaks. The magnetic space group lets
Co1 point in any direction and Co2 only along b.

In [22]:
structure.atom_site_moment['Co1'].crystalaxis_x.free = True
structure.atom_site_moment['Co1'].crystalaxis_y.free = True
structure.atom_site_moment['Co1'].crystalaxis_z.free = True
structure.atom_site_moment['Co2'].crystalaxis_y.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 41 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       13.06        48.28
      1      0.011     10.60        31.78   34.2% ↓
      2      0.016      7.03        13.99   56.0% ↓
      3      0.022      6.74        12.85   8.1% ↓
      4      0.027      6.04        10.33   19.6% ↓
      5      0.032      5.58         8.80   14.8% ↓
      6      0.039      5.42         8.32   5.5% ↓
      7      0.045      5.26         7.84   5.8% ↓
      8      0.050      5.09         7.33   6.4% ↓
      9      0.056      4.91         6.84   6.8% ↓
     10      0.061      4.90         6.80   0.5% ↓
     11      0.067      4.87         6.72   1.2% ↓
     12      0.077      4.87         6.71   0.1% ↓
     13      0.084      4.87         6.71   
     14      0.092      4.87         6.71   
     15      0.099      4.87         6.71   
     16      0.106      4.87         6.71   
     17      0.113      4.87  

View(html=<119,600 characters>, height=376)

In [24]:
project.display.pattern('d20')

View(html=<307,359 characters>, height=480)

### Magnetic form factor

Last, the j2 term of the Co²⁺ form factor, which changes how the magnetic intensities fall with
angle. It moves together with the moments, so it comes after them.

In [25]:
structure.magnetic_form_factor['Co'].j2_coefficient.free = True

In [26]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_cosio-d20_2k, 1438 pts fitted / 1506 loaded, 42 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        4.87         6.72
      1      0.008      4.72         6.30   6.2% ↓
      2      0.014      4.70         6.26   0.6% ↓
      3      0.023      4.70         6.25   0.3% ↓
      4      0.032      4.70         6.25   
      5      0.038      4.70         6.25   
      6      0.045      4.70         6.25   
      7      0.051      4.70         6.25   
      8      0.058      4.70         6.25   
      9      0.064      4.70         6.25   
     10      0.071      4.70         6.25   
     11      0.078      4.70         6.25   
     12      0.084      4.70         6.25   
     13      0.091      4.70         6.25   
     14      0.097      4.70         6.25   
     15      0.104      4.70         6.25   
     16      0.110      4.70         6.25   
     17      0.119      4.70         6.25   
  done · converged · 17 iters · reduced χ² 

View(html=<120,722 characters>, height=376)

In [27]:
project.display.pattern('d20')

View(html=<307,471 characters>, height=480)

In [28]:
project.display.fit.correlations()

,1,2,3,4,5
1 d20.instrument.calib_twotheta_offset,,,,,
2 d20.peak.broad_gauss_u,-0.24,,,,
3 d20.peak.broad_gauss_v,0.32,-0.94,,,
4 d20.peak.broad_gauss_w,-0.30,0.85,-0.93,,
5 d20.peak.asym_fcj_1,0.78,-0.30,0.40,-0.38,


In [29]:
project.display.structure('cosio')

View(html=<338,116 characters>, height=660)

## Save the project

In [30]:
project.save_as('pd-neut-cwl_cosio-d20_2k')
project.report.save_html('pd-neut-cwl_cosio-d20_2k.html')

Saving project to 'pd-neut-cwl_cosio-d20_2k'
Saving report to 'pd-neut-cwl_cosio-d20_2k.html'
